# AI Meeting Buddy - Sidebar Navigation Update

In this notebook, we'll update the sidebar navigation of the AI Meeting Buddy dashboard using shadcn/ui's Sidebar and Navigation components to create a meeting-focused sidebar with relevant navigation options.

## Requirements

1. Remove template items like Lifecycle, Analytics, Projects, Documents, Reports, Word Assistant
2. Include only meeting-relevant options:
   - Dashboard → overview page with meeting metrics, trust score, reminders
   - Agenda → pre-meeting prep, past notes, sales highlights
   - Scheduling → auto-find best slots, calendar integration
   - Follow-ups → track tasks, decisions, status
   - Reminders → upcoming deadlines and action alerts
   - Data Library → past meetings, sales records, notes (reference only)
   - Settings → account/profile configuration
   - Keep Get Help and Search in footer section
3. UI Style:
   - Use Sidebar + SidebarItem components from shadcn/ui
   - Each item should have a proper icon from lucide-react
   - Sidebar must be collapsible with hover tooltips when collapsed
   - Active item should have a highlight
   - Mobile responsive: collapses into a drawer

In [ ]:
"use client"

import * as React from "react"
import Link from "next/link"
import { usePathname } from "next/navigation"
import {
  LayoutDashboard,
  Notebook,
  Calendar,
  ListTodo,
  Bell,
  Archive,
  Settings,
  Search,
  HelpCircle,
  ChevronRight,
  ChevronLeft,
  Menu
} from "lucide-react"

import { useIsMobile } from "@/hooks/use-mobile"
import { cn } from "@/lib/utils"
import { Button } from "@/components/ui/button"
import {
  Sidebar,
  SidebarContent,
  SidebarFooter,
  SidebarHeader,
  SidebarMenu,
  SidebarMenuItem,
  SidebarMenuButton
} from "@/components/ui/sidebar"
import {
  Tooltip,
  TooltipContent,
  TooltipProvider,
  TooltipTrigger
} from "@/components/ui/tooltip"

// Navigation items configuration - meeting focused items
const navigationItems = [
  {
    title: "Dashboard",
    icon: LayoutDashboard,
    href: "/dashboard",
    description: "Overview with meeting metrics, trust score, reminders"
  },
  {
    title: "Agenda",
    icon: Notebook,
    href: "/agenda",
    description: "Pre-meeting prep, past notes, sales highlights"
  },
  {
    title: "Scheduling",
    icon: Calendar,
    href: "/scheduling",
    description: "Auto-find best slots, calendar integration"
  },
  {
    title: "Follow-ups",
    icon: ListTodo,
    href: "/follow-ups",
    description: "Track tasks, decisions, status"
  },
  {
    title: "Reminders",
    icon: Bell,
    href: "/reminders",
    description: "Upcoming deadlines and action alerts"
  },
  {
    title: "Data Library",
    icon: Archive,
    href: "/data-library",
    description: "Past meetings, sales records, notes"
  }
]

// Footer items
const footerItems = [
  {
    title: "Settings",
    icon: Settings,
    href: "/settings",
    description: "Account and profile configuration"
  },
  {
    title: "Search",
    icon: Search,
    href: "/search",
    description: "Find content across the app"
  },
  {
    title: "Get Help",
    icon: HelpCircle,
    href: "/help",
    description: "Support and documentation"
  }
]

## Step 1: Define Navigation Items and Icons

Above, we've defined the navigation items for our sidebar using lucide-react icons as specified in the requirements.

Now let's implement the SidebarItem component that will handle both the expanded and collapsed states with tooltips.

In [ ]:
// SidebarItem component for individual navigation items
interface SidebarItemProps {
  icon: React.ElementType
  title: string
  href: string
  description?: string
  isActive?: boolean
  isCollapsed: boolean
}

function SidebarItem({ icon: Icon, title, href, description, isActive, isCollapsed }: SidebarItemProps) {
  return (
    <SidebarMenuItem>
      {isCollapsed ? (
        <TooltipProvider>
          <Tooltip>
            <TooltipTrigger asChild>
              <SidebarMenuButton asChild className={cn("h-9 w-9", isActive && "bg-primary text-primary-foreground")}>
                <Link href={href}>
                  <Icon className="h-5 w-5" />
                  <span className="sr-only">{title}</span>
                </Link>
              </SidebarMenuButton>
            </TooltipTrigger>
            <TooltipContent side="right" className="border-none bg-primary/90 text-primary-foreground">
              <div>
                <p className="font-medium">{title}</p>
                {description && <p className="text-xs opacity-75">{description}</p>}
              </div>
            </TooltipContent>
          </Tooltip>
        </TooltipProvider>
      ) : (
        <SidebarMenuButton asChild className={cn("justify-start", isActive && "bg-primary text-primary-foreground")}>
          <Link href={href}>
            <Icon className="mr-2 h-5 w-5" />
            <span>{title}</span>
          </Link>
        </SidebarMenuButton>
      )}
    </SidebarMenuItem>
  )
}

## Step 2: Create the Main AppSidebar Component

Now, let's build our main AppSidebar component that will:
- Support collapsible behavior
- Be mobile responsive
- Properly style active navigation items 
- Group navigation items appropriately

In [ ]:
// Main AppSidebar component with collapsible behavior and mobile responsiveness
export function AppSidebar({ ...props }: React.ComponentProps<typeof Sidebar>) {
  const [isCollapsed, setIsCollapsed] = React.useState(false)
  const isMobile = useIsMobile()
  const pathname = usePathname()
  
  // Handle mobile view state
  const [isMobileMenuOpen, setIsMobileMenuOpen] = React.useState(false)

  return (
    <>
      {/* Mobile menu button - only shown on mobile */}
      {isMobile && (
        <Button 
          variant="ghost" 
          size="icon" 
          className="fixed top-4 left-4 z-50 md:hidden"
          onClick={() => setIsMobileMenuOpen(!isMobileMenuOpen)}
        >
          <Menu className="h-5 w-5" />
          <span className="sr-only">Toggle menu</span>
        </Button>
      )}
      
      {/* Sidebar component - visible based on state and device */}
      <Sidebar
        className={cn(
          "border-r transition-all duration-300",
          isCollapsed ? "w-[68px]" : "w-[240px]",
          isMobile && !isMobileMenuOpen ? "hidden" : "block",
          isMobile && isMobileMenuOpen ? "absolute inset-y-0 left-0 z-40" : ""
        )}
        {...props}
      >
        <SidebarHeader className="p-2">
          <div className="flex items-center justify-between px-3 py-2">
            {!isCollapsed && <h2 className="text-lg font-semibold">Meeting Buddy</h2>}
            
            {/* Toggle button for collapsing sidebar - hidden on mobile */}
            {!isMobile && (
              <Button 
                variant="ghost" 
                size="icon" 
                onClick={() => setIsCollapsed(!isCollapsed)}
                aria-label={isCollapsed ? "Expand sidebar" : "Collapse sidebar"}
              >
                {isCollapsed ? <ChevronRight className="h-4 w-4" /> : <ChevronLeft className="h-4 w-4" />}
              </Button>
            )}
          </div>
        </SidebarHeader>
        
        <SidebarContent className="p-2">
          <SidebarMenu>
            {/* Main navigation items */}
            {navigationItems.map((item) => (
              <SidebarItem
                key={item.href}
                icon={item.icon}
                title={item.title}
                href={item.href}
                description={item.description}
                isActive={pathname === item.href}
                isCollapsed={isCollapsed}
              />
            ))}
          </SidebarMenu>
          
          {/* Spacer */}
          <div className="my-4" />
          
          {/* Footer navigation items */}
          <SidebarMenu>
            {footerItems.map((item) => (
              <SidebarItem
                key={item.href}
                icon={item.icon}
                title={item.title}
                href={item.href}
                description={item.description}
                isActive={pathname === item.href}
                isCollapsed={isCollapsed}
              />
            ))}
          </SidebarMenu>
        </SidebarContent>
        
        <SidebarFooter className="p-2">
          {/* User profile could be added here */}
        </SidebarFooter>
      </Sidebar>
      
      {/* Overlay to close mobile menu when clicking outside */}
      {isMobile && isMobileMenuOpen && (
        <div 
          className="fixed inset-0 z-30 bg-background/80 backdrop-blur-sm"
          onClick={() => setIsMobileMenuOpen(false)}
        />
      )}
    </>
  )
}

## Step 3: Integration with Dashboard Page

Now let's update our app-sidebar.tsx file and integrate it with the dashboard layout:

In [ ]:
// File: components/app-sidebar.tsx
"use client"

import * as React from "react"
import Link from "next/link"
import { usePathname } from "next/navigation"
import {
  LayoutDashboard,
  Notebook,
  Calendar,
  ListTodo,
  Bell,
  Archive,
  Settings,
  Search,
  HelpCircle,
  ChevronRight,
  ChevronLeft,
  Menu
} from "lucide-react"

import { useIsMobile } from "@/hooks/use-mobile"
import { cn } from "@/lib/utils"
import { Button } from "@/components/ui/button"
import {
  Sidebar,
  SidebarContent,
  SidebarFooter,
  SidebarHeader,
  SidebarMenu,
  SidebarMenuItem,
  SidebarMenuButton
} from "@/components/ui/sidebar"
import {
  Tooltip,
  TooltipContent,
  TooltipProvider,
  TooltipTrigger
} from "@/components/ui/tooltip"

// Navigation items configuration - meeting focused items
const navigationItems = [
  {
    title: "Dashboard",
    icon: LayoutDashboard,
    href: "/dashboard",
    description: "Overview with meeting metrics, trust score, reminders"
  },
  {
    title: "Agenda",
    icon: Notebook,
    href: "/agenda",
    description: "Pre-meeting prep, past notes, sales highlights"
  },
  {
    title: "Scheduling",
    icon: Calendar,
    href: "/scheduling",
    description: "Auto-find best slots, calendar integration"
  },
  {
    title: "Follow-ups",
    icon: ListTodo,
    href: "/follow-ups",
    description: "Track tasks, decisions, status"
  },
  {
    title: "Reminders",
    icon: Bell,
    href: "/reminders",
    description: "Upcoming deadlines and action alerts"
  },
  {
    title: "Data Library",
    icon: Archive,
    href: "/data-library",
    description: "Past meetings, sales records, notes"
  }
]

// Footer items
const footerItems = [
  {
    title: "Settings",
    icon: Settings,
    href: "/settings",
    description: "Account and profile configuration"
  },
  {
    title: "Search",
    icon: Search,
    href: "/search",
    description: "Find content across the app"
  },
  {
    title: "Get Help",
    icon: HelpCircle,
    href: "/help",
    description: "Support and documentation"
  }
]

// SidebarItem component for individual navigation items
interface SidebarItemProps {
  icon: React.ElementType
  title: string
  href: string
  description?: string
  isActive?: boolean
  isCollapsed: boolean
}

function SidebarItem({ icon: Icon, title, href, description, isActive, isCollapsed }: SidebarItemProps) {
  return (
    <SidebarMenuItem>
      {isCollapsed ? (
        <TooltipProvider>
          <Tooltip>
            <TooltipTrigger asChild>
              <SidebarMenuButton asChild className={cn("h-9 w-9", isActive && "bg-primary text-primary-foreground")}>
                <Link href={href}>
                  <Icon className="h-5 w-5" />
                  <span className="sr-only">{title}</span>
                </Link>
              </SidebarMenuButton>
            </TooltipTrigger>
            <TooltipContent side="right" className="border-none bg-primary/90 text-primary-foreground">
              <div>
                <p className="font-medium">{title}</p>
                {description && <p className="text-xs opacity-75">{description}</p>}
              </div>
            </TooltipContent>
          </Tooltip>
        </TooltipProvider>
      ) : (
        <SidebarMenuButton asChild className={cn("justify-start", isActive && "bg-primary text-primary-foreground")}>
          <Link href={href}>
            <Icon className="mr-2 h-5 w-5" />
            <span>{title}</span>
          </Link>
        </SidebarMenuButton>
      )}
    </SidebarMenuItem>
  )
}

// Main AppSidebar component
export function AppSidebar({ ...props }: React.ComponentProps<typeof Sidebar>) {
  const [isCollapsed, setIsCollapsed] = React.useState(false)
  const isMobile = useIsMobile()
  const pathname = usePathname()
  
  // Handle mobile view state
  const [isMobileMenuOpen, setIsMobileMenuOpen] = React.useState(false)

  return (
    <>
      {/* Mobile menu button - only shown on mobile */}
      {isMobile && (
        <Button 
          variant="ghost" 
          size="icon" 
          className="fixed top-4 left-4 z-50 md:hidden"
          onClick={() => setIsMobileMenuOpen(!isMobileMenuOpen)}
        >
          <Menu className="h-5 w-5" />
          <span className="sr-only">Toggle menu</span>
        </Button>
      )}
      
      {/* Sidebar component - visible based on state and device */}
      <Sidebar
        className={cn(
          "border-r transition-all duration-300",
          isCollapsed ? "w-[68px]" : "w-[240px]",
          isMobile && !isMobileMenuOpen ? "hidden" : "block",
          isMobile && isMobileMenuOpen ? "absolute inset-y-0 left-0 z-40" : ""
        )}
        {...props}
      >
        <SidebarHeader className="p-2">
          <div className="flex items-center justify-between px-3 py-2">
            {!isCollapsed && <h2 className="text-lg font-semibold">Meeting Buddy</h2>}
            
            {/* Toggle button for collapsing sidebar - hidden on mobile */}
            {!isMobile && (
              <Button 
                variant="ghost" 
                size="icon" 
                onClick={() => setIsCollapsed(!isCollapsed)}
                aria-label={isCollapsed ? "Expand sidebar" : "Collapse sidebar"}
              >
                {isCollapsed ? <ChevronRight className="h-4 w-4" /> : <ChevronLeft className="h-4 w-4" />}
              </Button>
            )}
          </div>
        </SidebarHeader>
        
        <SidebarContent className="p-2">
          <SidebarMenu>
            {/* Main navigation items */}
            {navigationItems.map((item) => (
              <SidebarItem
                key={item.href}
                icon={item.icon}
                title={item.title}
                href={item.href}
                description={item.description}
                isActive={pathname === item.href}
                isCollapsed={isCollapsed}
              />
            ))}
          </SidebarMenu>
          
          {/* Spacer */}
          <div className="my-4" />
          
          {/* Footer navigation items */}
          <SidebarMenu>
            {footerItems.map((item) => (
              <SidebarItem
                key={item.href}
                icon={item.icon}
                title={item.title}
                href={item.href}
                description={item.description}
                isActive={pathname === item.href}
                isCollapsed={isCollapsed}
              />
            ))}
          </SidebarMenu>
        </SidebarContent>
        
        <SidebarFooter className="p-2">
          {/* User profile could be added here */}
        </SidebarFooter>
      </Sidebar>
      
      {/* Overlay to close mobile menu when clicking outside */}
      {isMobile && isMobileMenuOpen && (
        <div 
          className="fixed inset-0 z-30 bg-background/80 backdrop-blur-sm"
          onClick={() => setIsMobileMenuOpen(false)}
        />
      )}
    </>
  )
}

In [ ]:
// Example integration with dashboard page layout
// File: app/dashboard/page.tsx

import { AppSidebar } from "@/components/app-sidebar"

export default function DashboardPage() {
  return (
    <div className="flex min-h-screen">
      <AppSidebar />
      <div className="flex-1 p-6">
        <h1 className="text-2xl font-bold">Dashboard</h1>
        <p className="text-muted-foreground mt-2">Overview of your meetings, trust scores, and reminders</p>
        
        {/* Dashboard content */}
        <div className="grid gap-4 md:grid-cols-2 lg:grid-cols-3 mt-6">
          {/* Dashboard cards would go here */}
        </div>
      </div>
    </div>
  )
}

## Implementation Summary

This new sidebar navigation implementation meets all the requirements:

1. ✅ **Meeting-focused navigation items**: 
   - Dashboard for meeting metrics and reminders
   - Agenda for meeting preparation and notes
   - Scheduling for calendar integration
   - Follow-ups for tracking tasks and decisions
   - Reminders for deadlines and alerts
   - Data Library for past meeting records
   - Settings, Get Help, and Search maintained in the footer

2. ✅ **Modern UI with shadcn/ui components**:
   - Uses Sidebar and related components from shadcn/ui
   - Each item has an appropriate lucide-react icon
   - Active items get highlighted with primary background color
   - Tooltips appear when hovering over collapsed icons

3. ✅ **Collapsible behavior**:
   - Toggle button to collapse/expand the sidebar
   - Collapsed state shows only icons
   - Tooltips provide context in collapsed state

4. ✅ **Mobile responsiveness**:
   - Collapses into a drawer on mobile devices
   - Hamburger menu button for toggling visibility
   - Backdrop overlay when mobile menu is open

To implement this in your project:
1. Replace the content of `components/app-sidebar.tsx` with the complete code from above
2. Update your dashboard layout to use the new sidebar
3. Ensure you have the required dependencies and hooks (like useIsMobile)